# Physics 110A (Regan) — Problem Set #2: Electrostatics

Sequel to `regan_ps1_relativity.ipynb`. Griffiths Ch. 2, the field $\mathbf E$ from charge — first by
**brute-force Coulomb integration** (to "build strength and fortitude"), then the same results the
easy way with **Gauss's law**, plus superposition and the curl test for impossible fields.

| # | problem | method | where |
|---|---------|--------|-------|
| 1 | G2.5 field above a ring | Coulomb, on-axis | §1 |
| 2 | G2.7 shell, the hard way | Coulomb + law of cosines | §2 |
| 3 | G2.8 solid sphere as shells | superpose §2 | §3 |
| 4 | G2.10 charge in a cube corner | symmetry (no calculus) | §4 |
| 5 | G2.11 shell, Gauss's law | Gauss | §5 |
| 6 | G2.12 solid sphere, Gauss's law | Gauss | §5 |
| 7 | G2.13 line charge | Gauss (cylinder) | §6 |
| 8 | G2.18 overlapping spheres | superposition | §7 |
| 9 | G2.20 impossible fields | curl test | §8 |
| 10 | G2.22 potential of a line | line integral | §9 |

Engine: `griffiths.electrostatics` (added for this set). $\epsilon_0$ stays symbolic;
$k_e = 1/4\pi\epsilon_0$.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import sympy as sp
from sympy import oo, pi
from IPython.display import display, Math

import griffiths as gr
from griffiths import electrostatics as es
from griffiths import x, y, z

sp.init_printing(use_latex="mathjax")
lam, R, Z, sigma, Q, rho, a = sp.symbols("lambda R Z sigma Q rho a", positive=True)
r, s, k = sp.symbols("r s k", positive=True)
print("griffiths.electrostatics loaded; k_e =", sp.latex(es.KE))

griffiths.electrostatics loaded; k_e = \frac{1}{4 \pi \epsilon_{0}}


## §1 — G2.5: field on the axis of a charged ring

A ring of radius $R$, line charge $\lambda$, field a height $Z$ above its centre. Horizontal
components cancel by symmetry; what survives is

$$E_z = \frac{1}{4\pi\epsilon_0}\!\int\frac{\lambda\,dl}{\mathscr r^2}\cos\theta,
\qquad \mathscr r^2 = R^2+Z^2,\ \cos\theta=\frac{Z}{\mathscr r},\ \oint dl = 2\pi R.$$

In [2]:
E = es.ring_field_axis(lam, R, Z)
display(Math(r"E_z = " + sp.latex(E) + r"\,\hat z"))

# the requested limit checks
display(Math(r"Z\to 0\ (\text{centre}):\quad E_z \to " + sp.latex(sp.limit(E, Z, 0))
             + r"\quad\text{(symmetry: zero at the centre)}"))
q_ring = lam * 2 * pi * R
display(Math(r"Z\gg R:\quad E_z \to " + sp.latex(sp.simplify(E.subs(R, 0)))
             + r"\ \to\ \frac{1}{4\pi\epsilon_0}\frac{q}{Z^2}"
             r"\quad (q=\lambda\,2\pi R,\ \text{point charge})"))
ratio = sp.limit(E / (es.KE * q_ring / Z**2), Z, oo)
print("E_z / [k q/Z^2] as Z->oo =", ratio, " (-> 1: recovers the point charge)")

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

E_z / [k q/Z^2] as Z->oo = 1  (-> 1: recovers the point charge)


## §2 — G2.7: spherical shell the *hard* way (Coulomb's law)

Observation point on the axis a distance $Z$ from the centre of a shell (radius $R$, surface charge
$\sigma$). With the law of cosines $\mathscr r^2 = R^2+Z^2-2RZ\cos\theta'$ (a dot product in
disguise), the substitution $u=\cos\theta'$ makes the integral elementary. The payoff: the shell
acts like a point charge outside and contributes **nothing** inside.

In [3]:
E_out = es.shell_field_coulomb(sigma, R, Z, "outside")
E_in = es.shell_field_coulomb(sigma, R, Z, "inside")
q_shell = sigma * 4 * pi * R**2
display(Math(r"Z>R:\quad E = " + sp.latex(E_out)
             + r" = \frac{1}{4\pi\epsilon_0}\frac{q}{Z^2}\quad(q=4\pi R^2\sigma)"))
display(Math(r"Z<R:\quad E = " + sp.latex(E_in) + r"\quad\text{(field-free interior)}"))
assert sp.simplify(E_out - es.KE * q_shell / Z**2) == 0

<IPython.core.display.Math object>

<IPython.core.display.Math object>

## §3 — G2.8: solid sphere as a stack of shells

A uniform solid sphere ($\rho$, radius $a$) is nested shells of charge $dq=\rho\,4\pi R^2\,dR$. Each
shell at radius $R$ contributes $k_e\,dq/Z^2$ if $R<Z$ and nothing if $R>Z$ (the §2 result). Summing:

In [4]:
# field at radius Z by superposing shells (each shell: point-charge if inside, 0 if outside)
shell_dq = rho * 4 * pi * R**2
# inside the sphere (Z < a): only shells with R < Z contribute
E_inside = sp.simplify(sp.integrate(es.KE * shell_dq / Z**2, (R, 0, Z)))
# outside (Z > a): all shells contribute
E_outside = sp.simplify(sp.integrate(es.KE * shell_dq / Z**2, (R, 0, a)))
display(Math(r"Z<a:\quad E = " + sp.latex(E_inside)
             + r" = \frac{\rho Z}{3\epsilon_0} = \frac{1}{4\pi\epsilon_0}\frac{Q Z}{a^3}"))
display(Math(r"Z>a:\quad E = " + sp.latex(E_outside)
             + r" = \frac{1}{4\pi\epsilon_0}\frac{Q}{Z^2}\quad(Q=\tfrac{4}{3}\pi a^3\rho)"))
Qtot = rho * sp.Rational(4, 3) * pi * a**3
assert sp.simplify(E_outside - es.KE * Qtot / Z**2) == 0

<IPython.core.display.Math object>

<IPython.core.display.Math object>

## §4 — G2.10: a charge in the corner of a cube (no calculus)

Pure symmetry. Surround the corner charge $q$ by **eight** identical cubes meeting at that corner;
together they enclose $q$, so by Gauss each cube captures a flux $q/8\epsilon_0$. Within one cube the
three faces *touching* the corner carry zero flux (the field grazes them), so the whole $q/8\epsilon_0$
exits through the **three far faces** — sharing equally,

$$\Phi_{\text{one far face}} = \frac{q}{24\,\epsilon_0}.$$

In [5]:
eps0 = es.eps0; q = sp.Symbol("q", positive=True)
total = q / eps0
per_cube = total / 8                         # 8 cubes share the charge
per_far_face = per_cube / 3                  # 3 far faces share each cube's flux
display(Math(r"\Phi_{\text{cube}} = \frac{q}{8\epsilon_0} = " + sp.latex(per_cube)
             + r",\qquad \Phi_{\text{far face}} = \frac{q}{24\epsilon_0} = "
             + sp.latex(per_far_face)))
print("touching faces carry zero flux; 3 far faces share the cube's q/8e0 equally.")

<IPython.core.display.Math object>

touching faces carry zero flux; 3 far faces share the cube's q/8e0 equally.


## §5 — G2.11 & G2.12: shell and solid sphere by Gauss's law

The easy way. A Gaussian sphere of radius $r$ sees only the **enclosed** charge:
$E\,4\pi r^2 = Q_{\text{enc}}/\epsilon_0$. Same answers as §2–§3, in two lines instead of an integral.

In [6]:
# G2.11 shell
E_shell = es.gauss_sphere(Q, r, R=R, uniform=False)
display(Math(r"\text{shell:}\quad E(r) = " + sp.latex(E_shell)))
# G2.12 solid uniform sphere
E_solid = es.gauss_sphere(Q, r, R=a, uniform=True)
display(Math(r"\text{solid sphere:}\quad E(r) = " + sp.latex(E_solid)))
print("inside grows linearly (kQr/a^3), outside falls as kQ/r^2 -- continuous at r=a.")

<IPython.core.display.Math object>

<IPython.core.display.Math object>

inside grows linearly (kQr/a^3), outside falls as kQ/r^2 -- continuous at r=a.


## §6 — G2.13: infinite line charge by Gauss's law

A coaxial Gaussian cylinder (radius $s$, length $L$): flux only through the curved wall,
$E\,(2\pi s L) = \lambda L/\epsilon_0$.

In [7]:
E_line = es.gauss_line(lam, s)
display(Math(r"E_s = " + sp.latex(E_line) + r"\,\hat s \;=\; \frac{\lambda}{2\pi\epsilon_0 s}\,\hat s"))

<IPython.core.display.Math object>

## §7 — G2.18: field where two uniform spheres overlap

A $+\rho$ sphere and a $-\rho$ sphere, centres offset by $\mathbf d$. *Don't* pick a coordinate
system — superpose. Inside a uniform sphere $\mathbf E=\rho\,\mathbf r/3\epsilon_0$ (from §5,
pointing from its centre). At any overlap point, $\mathbf E_+ + \mathbf E_- =
\frac{\rho}{3\epsilon_0}(\mathbf r_+-\mathbf r_-) = \frac{\rho}{3\epsilon_0}\mathbf d$ — a
**uniform** field, independent of position.

In [7]:
dx, dy, dz = sp.symbols("d_x d_y d_z", real=True)
d = sp.Matrix([dx, dy, dz])
E_overlap = es.overlapping_spheres_field(rho, d)
display(Math(r"\mathbf E_{\text{overlap}} = \frac{\rho}{3\epsilon_0}\,\mathbf d = "
             + sp.latex(E_overlap.T) + r"\quad\text{(constant in the overlap)}"))
print("the position cancels: r_+ - r_- = d for every point -> a uniform interior field.")

<IPython.core.display.Math object>

the position cancels: r_+ - r_- = d for every point -> a uniform interior field.


## §8 — G2.20: which fields could be electrostatic?

A static $\mathbf E$ must be curl-free ($\nabla\times\mathbf E=0$, so that $\mathbf E=-\nabla V$).
Test the two candidates; for the admissible one, recover $V$ by line integration.

In [8]:
E_a = sp.Matrix([k*x*y, k*2*y*z, k*3*x*z])
E_b = sp.Matrix([k*y**2, k*(2*x*y + z**2), k*2*y*z])
for name, E in [("(a)", E_a), ("(b)", E_b)]:
    ok, curlE = es.is_electrostatic(E)
    display(Math(rf"\text{{{name}}}\ \nabla\times\mathbf E = " + sp.latex(curlE.T)
                 + (r"\;=\;0\ \Rightarrow\ \textbf{allowed}" if ok
                    else r"\;\ne\;0\ \Rightarrow\ \textbf{impossible}")))
V = es.potential_of(E_b)
display(Math(r"\text{(b)}\quad V = " + sp.latex(V)
             + r"\quad\text{with}\quad \mathbf E = -\nabla V\ \checkmark"))
assert sp.simplify(sp.Matrix([-sp.diff(V, v) for v in (x, y, z)]) - E_b) == sp.zeros(3, 1)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

## §9 — G2.22: potential of an infinite line charge

$V(s) = -\int_{s_{\text{ref}}}^{s}\mathbf E\cdot d\mathbf l$. The catch: for an infinite line the
field falls as $1/s$, so the integral *diverges* if you put the reference at infinity — you must pick
a finite reference distance $s_{\text{ref}}$.

In [ ]:
s_ref = sp.Symbol("s_ref", positive=True)
V_line = es.line_charge_potential(lam, s, s_ref)
display(Math(r"V(s) = -\int_{s_{\text{ref}}}^{s} \frac{\lambda}{2\pi\epsilon_0 s'}\,ds' = "
             + sp.latex(V_line) + r" = \frac{\lambda}{2\pi\epsilon_0}\ln\!\frac{s_{\text{ref}}}{s}"))
# consistency: -dV/ds returns the field
display(Math(r"-\frac{dV}{ds} = " + sp.latex(sp.simplify(-sp.diff(V_line, s)))
             + r" = E_s\ \checkmark"))
display(Math(r"\text{ref at }\infty:\ V \to " + r"\text{diverges}"
             + r"\quad(\text{can't use } s_{\text{ref}}=\infty\text{ for an infinite line})"))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

: 

## §10 — Connections

- **Hard-way Coulomb vs Gauss**: §2–§3 grind the shell/sphere integrals out by brute force; §5 gets
  the same answers from symmetry in two lines. The lesson Regan is after — symmetry is leverage —
  is the same one that makes the FFT (and the dispersion bookkeeping in `gs_core.py`) fast.
- **Superposition** (§3, §7) is linearity, the property the whole repo's linear-optics pipeline and
  the FNO depend on.
- **Curl-free $\Rightarrow$ a potential exists** (§8) is the electrostatic face of the conservative-
  field test in `gradcurldiv_jalali_robotics.ipynb`; the line-charge potential's $\ln s$ is the 2-D
  Green's function of $\nabla^2$.

Engine added this session: `griffiths/electrostatics.py` (ring/disk/shell Coulomb integrals, Gauss
fields, curl test, line potential, overlapping spheres), benchmarked in
`scripts/smoke_electrostatics.py`. PS#1 lives in `regan_ps1_relativity.ipynb`.